# Baseline age model: train and evaluate on validation (tutorial)

This notebook is for **competitors and teachers** learning how the repository’s baseline pipeline works.

**What you will do**
1. Check that split pseudobulk files and label CSVs exist (`train` + `val`).
2. Train the baseline Random Forest with `models/train_age_model_v2.py`.
3. Evaluate predictions against **validation** truth with `models/evaluate_val.py`.

**How to run**
- Open this notebook from the project root `aging-challenge-2026/`, or the first code cell sets paths relative to the repo.
- Execute cells **in order** (top to bottom).

**Time / resources**  
- Training uses CPU RAM and can take several minutes depending on `--n-genes` and `--n-estimators`. On a login node, use modest settings first (defaults are reasonable).

## 1. Concepts: what the baseline does

### Input data
- **Files:** split donor-aggregated pseudobulk under `data/scRNA-seq_pseudobulk/`:
  - `train_pseudobulk_donor_aggregated_public.h5ad`
  - `val_pseudobulk_donor_aggregated_public.h5ad`
- **Rows:** one row per **donor** (not per cell).
- **Columns (features):** for each gene, there are **five** pseudobulk expression values—one per **cell-type group** (e.g. CD4 T, CD8 T, NK, B, Monocytes). Feature names look like `ENSG...__CellTypeGroup`.
- `age` is intentionally stripped from these public pseudobulk h5ad files. Training/evaluation labels come from CSV files:
  - `data/metadata/train_age.csv`
  - `data/metadata/val_age.csv`
- This notebook uses separate train/val h5ad files directly (no merged temporary file).

### Training script (`train_age_model_v2.py`)
- **Model:** scikit-learn `RandomForestRegressor` (regression → predicted age in years).
- **Input:** separate train/val pseudobulk h5ad files + train/val label CSV files.
- **Feature selection (default):** keeps the top **N genes by variance** across train donors.
- **Alternative:** `--all-features` uses every gene (much heavier).
- **Preprocessing:** `log1p` on pseudobulk counts.
- **Output:** `val_predictions.csv`, `val_metrics.csv`, model and top features.

### Evaluation script (`evaluate_val.py`)
- Merges predictions with **validation** ground-truth ages by `donor_id`.
- **Ground-truth source:** `data/metadata/val_age.csv` (`donor_id`, `age`).
- Reports **MAE**, **RMSE**, **R²**, **Pearson**, **Spearman** and writes `val_eval_metrics.csv` next to the predictions file.

## 2. Environment: paths and Python

We locate the **repository root** so subprocess calls match `python models/train_age_model_v2.py` from the project root.

**Teaching note:** In HPC environments, activate the same conda/env that has `scanpy`, `sklearn`, `scipy` installed before starting Jupyter.

In [ ]:
import sys
import subprocess
from pathlib import Path

# If the notebook lives in notebooks/, parents[1] is the repo root
NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model_v2.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model_v2.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model_v2.py. Open Jupyter from the repo root or from notebooks/."
    )

DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Python:", sys.executable)

## 3. Check prerequisites

Before training, you need split donor-aggregated pseudobulk files:
- `data/scRNA-seq_pseudobulk/train_pseudobulk_donor_aggregated_public.h5ad`
- `data/scRNA-seq_pseudobulk/val_pseudobulk_donor_aggregated_public.h5ad`

For labels, you need:
- `data/metadata/train_age.csv` (columns: `donor_id,age`)
- `data/metadata/val_age.csv` (columns: `donor_id,age`)

In [ ]:
import pandas as pd

PB_DIR = DATA_DIR / "scRNA-seq_pseudobulk"
META_DIR = DATA_DIR / "metadata"
TRAIN_H5AD = PB_DIR / "train_pseudobulk_donor_aggregated_public.h5ad"
VAL_H5AD   = PB_DIR / "val_pseudobulk_donor_aggregated_public.h5ad"
TRAIN_AGE_CSV = META_DIR / "train_age.csv"
VAL_AGE_CSV   = META_DIR / "val_age.csv"

print("Train pseudobulk:", TRAIN_H5AD, "exists=", TRAIN_H5AD.exists())
print("Val pseudobulk  :", VAL_H5AD,   "exists=", VAL_H5AD.exists())
print("Train age CSV   :", TRAIN_AGE_CSV, "exists=", TRAIN_AGE_CSV.exists())
print("Val age CSV     :", VAL_AGE_CSV,   "exists=", VAL_AGE_CSV.exists())

if not (TRAIN_H5AD.exists() and VAL_H5AD.exists()):
    raise FileNotFoundError(
        "Need both train and val donor-aggregated h5ad files under data/scRNA-seq_pseudobulk/."
    )
if not (TRAIN_AGE_CSV.exists() and VAL_AGE_CSV.exists()):
    raise FileNotFoundError("Need train_age.csv and val_age.csv under data/metadata/.")

for p, name in [(TRAIN_AGE_CSV, 'train_age.csv'), (VAL_AGE_CSV, 'val_age.csv')]:
    df = pd.read_csv(p)
    if not {'donor_id', 'age'}.issubset(df.columns):
        raise ValueError(f"{name} must contain columns donor_id, age")

TRUTH_PATH = VAL_AGE_CSV
print("\nUsing separate train/val files directly (no merged h5ad).")

## 4. Run training (`train_age_model_v2.py`)

This step trains the model and writes predictions. Evaluation is intentionally separated into Step 6.

```bash
python models/train_age_model_v2.py --train-h5ad <train.h5ad> --val-h5ad <val.h5ad> --train-labels <train_age.csv> --val-labels <val_age.csv> [options]
```

How training works:
- Loads train and val donor-level feature matrices from separate h5ad files.
- Loads train/val labels from CSV and aligns by `donor_id`.
- If `ALL_FEATURES=False`, selects top `N_GENES` by variance on train features, keeping all 5 cell-type slots per selected gene.
- Applies `log1p` transform.
- Fits a Random Forest (`N_ESTIMATORS`, `SEED`, optional `MAX_DEPTH`).
- Writes `val_predictions.csv` and model artifacts in `results/<timestamp>/`.

### ML concepts behind key lines
- `def select_genes(...)`: feature-selection helper. It ranks genes by variance (how much they change across donors) and keeps the most informative ones. This reduces noise, memory use, and training time.
- `X_train = np.log1p(X_train)`: transforms counts with `log(1+x)`. This compresses very large values, reduces skew/outlier dominance, and usually improves tree-model stability on count-like features.
- `rf = RandomForestRegressor(...)`: creates an ensemble of decision trees.
  - `n_estimators`: number of trees (more trees = usually more stable, but slower)
  - `max_depth`: limit tree complexity (helps control overfitting)
  - `random_state`: reproducibility (same seed, same random choices)
  - `n_jobs=-1`: use all CPU cores available
- `rf.fit(X_train, y_train)`: training step. Trees learn patterns from features (`X_train`) to ages (`y_train`).
- `y_val_pred = rf.predict(X_val)`: inference step on unseen validation donors, producing age predictions used in Step 6 metrics.

**Parameters you can change below**
- `N_GENES` — number of highest-variance genes to keep (ignored if `ALL_FEATURES=True`).
- `N_ESTIMATORS` — number of trees.
- `ALL_FEATURES` — use all features (slower/heavier).
- `OUTPUT_DIR` — `None` for timestamped output, or fixed path for reproducible reruns.

Use Step 6 to compute/report validation metrics via `evaluate_val.py`.

In [ ]:
from datetime import datetime

# ---- Tune these for class demos / quick tests ----
N_GENES = 2000
N_ESTIMATORS = 100
ALL_FEATURES = False
SEED = 42
# Optional: set to a string path to pin outputs, e.g. str(RESULTS_DIR / "demo_run")
OUTPUT_DIR = None  # None → timestamped folder under results/

train_py = PROJ_ROOT / "models" / "train_age_model_v2.py"
cmd = [sys.executable, str(train_py)]
cmd += [
    "--train-h5ad", str(TRAIN_H5AD),
    "--val-h5ad", str(VAL_H5AD),
    "--train-labels", str(TRAIN_AGE_CSV),
    "--val-labels", str(VAL_AGE_CSV),
    "--n-genes", str(N_GENES),
    "--n-estimators", str(N_ESTIMATORS),
    "--seed", str(SEED),
]
if ALL_FEATURES:
    cmd.append("--all-features")
if OUTPUT_DIR:
    cmd += ["--output-dir", str(OUTPUT_DIR)]
else:
    run_dir = RESULTS_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    cmd += ["--output-dir", str(run_dir)]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
if result.returncode != 0:
    raise RuntimeError("train_age_model_v2.py exited with code %s" % result.returncode)
print("\nTraining finished successfully.")

## 5. Locate the latest run and preview predictions

By default, this notebook writes outputs to `results/YYYY-MM-DD_HH-MM-SS/` (via `--output-dir`).

For this validation-focused workflow, use `val_predictions.csv` from the run directory.

**Files written**
- `val_predictions.csv` — `donor_id`, `predicted_age` for validation donors
- `rf_age_model.joblib` — fitted model
- `top_genes.csv` — top genes by Random Forest **feature importance** (aggregated across cell-type columns)

In [ ]:
import pandas as pd
from IPython.display import display

out_base = RESULTS_DIR
subdirs = sorted([d for d in out_base.iterdir() if d.is_dir()], reverse=True)
run_dir = None
pred_path = None
for d in subdirs:
    p = d / "val_predictions.csv"
    if p.exists():
        run_dir = d
        pred_path = p
        break

if pred_path is None:
    raise FileNotFoundError("No val_predictions.csv found under results/")

print("Run directory:", run_dir)
print("Predictions file:", pred_path)
display(pd.read_csv(pred_path).head(10))

## 6. Run evaluation (`evaluate_val.py`)

This merges predictions with validation labels from `data/metadata/val_age.csv` and prints regression metrics.

**Teaching notes**
- This notebook does **not** require test labels.
- For `--plot`, matplotlib must be installed; the PNG is saved beside the predictions.
- You can pass an explicit `--predictions` path to score a specific run folder.

In [ ]:
eval_py = PROJ_ROOT / "models" / "evaluate_val.py"

cmd = [
    sys.executable,
    str(eval_py),
    "--predictions", str(pred_path),
    "--truth-csv", str(TRUTH_PATH),
]
# Uncomment to save scatter plot (requires matplotlib):
# cmd.append("--plot")
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
if result.returncode != 0:
    raise RuntimeError("evaluate_val.py exited with code %s" % result.returncode)

## 7. Load saved metrics

Metrics are stored as `val_eval_metrics.csv` next to the predictions file. **Interpret briefly:**
- **MAE** — average error in years.
- **R²** — fraction of age variance explained (higher is better, capped at 1 for perfect predictions).
- **Pearson / Spearman** — correlation between predicted and true age linearly vs. by rank.

In [ ]:
from IPython.display import display

metrics_file = pred_path.parent / "val_eval_metrics.csv"
if metrics_file.exists():
    display(pd.read_csv(metrics_file))
else:
    print("No val_eval_metrics.csv yet (run evaluation cell).")

## 8. Optional: top genes from the last training run

Feature importances are **not** causal; they indicate how much the forest used each gene’s pseudobulk signal in this pipeline. High overlap with biological ageing literature can be a sanity check.

In [ ]:
from IPython.display import display

top_path = pred_path.parent / "top_genes.csv"
if top_path.exists():
    display(pd.read_csv(top_path))
else:
    print("No top_genes.csv beside this run.")

## Further ideas for competitors

- Swap the Random Forest for another regressor (e.g. gradient boosting, elastic net on PCA).
- Engineer features (pathways, cell-type proportions) instead of raw pseudobulk.
- Use cell-level models + pooling; compare to this donor-aggregated baseline.
- Keep test labels hidden; tune only on train/validation labels.

See `models/README.md` and the main `README.md` for paths and data structure.